# Compare MOM–CICE–WW3 with standalone CICE6
The aim of this notebook is to compare a three-way coupled model (MCW) ouputs with those of standalone CICE6. The models compared are:
- MCW @100km w/ JRA55-do IAF
- CICE6-WIM @100km w/ JRA55-do IAF
- CICE6 @100km w/ JRA55-do IAF

In [ ]:
import xarray as xr
import cf_xarray
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client


import matplotlib.path as mpath
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.feature as cft
import cftime

# Stats
import geopandas as gpd
from scipy.interpolate import griddata

# Plotting
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import cmocean.cm as cmo

import os
import requests
import zipfile


from tqdm.notebook import tqdm 
import calendar
import pandas as pd
from datetime import datetime

# CLEANUP
%matplotlib inline
import seaborn as sns
import calendar

In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
print(client.dashboard_link)

### Read in MCW data

In [ ]:
### USER EDIT start
# esm_file = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta/experiment_datastore.json"
esm_file = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_iaf_2010/experiment_datastore.json"
dpi=300
### USER EDIT stop

import os
from matplotlib import rcParams
%matplotlib inline
rcParams["figure.dpi"]= dpi

plotfolder=f"/g/data/{os.environ['PROJECT']}/{os.environ['USER']}/access-om3-analysis-figs/Compare_MCW_CICE6"
os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ", esm_file)
print("Plot folder path: ", plotfolder)

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)


In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

datastore_filtered = datastore.search(realm="seaIce", frequency="1mon")

available_variables(datastore_filtered)

In [ ]:
ds_mcw = datastore.search(variable=["aice", "wave_sig_ht", "fsdrad"], 
                          frequency="1day", 
                          file_id='access_om3_cice_1day_mean_XXXX_XX'
                         ).to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"nj": -1, "ni": -1},
        decode_timedelta=True
    )
)
ds_grid = datastore.search(variable=["tarea", "HTE"], frequency="fx", realm="seaIce").to_dask().compute()
ds_grid

coords = datastore.search(variable=["TLAT", "TLON"], #file_id='access_om3_mom6_static'
                         ).to_dask().compute() # TODO why do we need file_id for my runs??
coords = coords.fillna(0.0)

ds_mcw = xr.merge([ds_mcw, ds_grid])
ds_mcw = ds_mcw.assign_coords(coords)
ds_mcw = ds_mcw.roll(ni=80, roll_coords=True)
ds_mcw

### Read in ACCESS-OM2 data

In [ ]:
exptname = '025deg_jra55_iaf_omip2_cycle1'
datastore = intake.cat.access_nri[exptname]
geolon = datastore.search(variable="geolon_t").to_dask().geolon_t
geolat = datastore.search(variable="geolat_t").to_dask().geolat_t
variable = "aice_m"
frequency = "1mon"

ds_om2 = datastore.search(variable=variable, frequency=frequency).to_dask()
# , frequency=frequency,
#                             # variable_cell_methods='.*time: max.*'
#                             ).to_dask(
#     xarray_open_kwargs = dict(
#         chunks={"time": -1},
#         decode_timedelta=True
#     ),
#     xarray_combine_by_coords_kwargs=dict(
#         compat="override",
#         data_vars="minimal",
#         coords="minimal"
#     )
# )[variable].cf.assign_coords({ "longitude": geolon, "latitude": geolat })

In [ ]:
ds_om2

In [ ]:
# datastore.search(variable=["aice", "wave_sig_ht", "fsdrad"], frequency="1day").unique().file_id

### Read in CICE6-WIM data

In [ ]:
# ["baseline-1deg", "wave-profile-1deg", "wave-propagation-1deg"],
import socket
from datetime import timedelta

def get_path(experiments):
    hostname = socket.gethostname()
    if "gadi" in hostname:
        machine = "gadi"
        #path = "/g/data/ps29/nd0349/runs/cice6/"
        path = "/scratch/ps29/nd0349/CICE_RUNS/"
        fig_path = "/home/566/nd0349/access-om3-analysis/figures/"
    elif "setonix" in hostname:
        machine = "setonix"
    else:
        machine = "noahday"
        path = "/Users/noahday/GitHub/cice-dev/cice-dirs/runs/"
        fig_path = "/Users/noahday/GitHub/access-om3-analysis/figures/"

    print(f"Running on {machine}")
        
    return path, fig_path


In [ ]:
histfreq = "d"
experiment = "wave-propagation-1deg-lognormal"
expt_path, fig_path = get_path(experiment)

path = os.path.join(expt_path, experiment, "history/")
if histfreq == "h":
    file_pattern = os.path.join(path, "iceh_01h.????-??-??-?????.nc")
    time_delta = timedelta(minutes=30)
elif histfreq == "d":
    file_pattern = os.path.join(path, "iceh.????-??-??.nc")
    time_delta = timedelta(hours=12)
elif histfreq == "m":
    file_pattern = os.path.join(path, "iceh.????-??.nc")  # Adjust this if needed
    time_delta = timedelta(days=16)
files = sorted(glob.glob(file_pattern))

if not files:
    print(f"❌ No files found for {experiment}, skipping...")

basic_ice_vars = ["aice", "hi", "iage", "fsdrad"]
atm_forcing_vars = ["Tair", "uatm", "vatm", "Qref", "fswdn", "flwdn", "snow"]
thermo_vars = ["meltb", "meltl", "meltt"]
# 10m Air temperature, wind components, 2m specific humiditiy, incoming long wave radiation, incoming long wave radiation
# Tair, uatm, vatm, Qa, fsw, flw, fsnow from ice_forcing.F90/JRA55_data

ocn_forcing_vars = ["sst", "sss", "uocn", "vocn"]
wave_forcing_vars = ["wave_sig_ht"] # peak period? MWD?

VARS = basic_ice_vars + atm_forcing_vars + ocn_forcing_vars + wave_forcing_vars + thermo_vars
VARS_with_m = VARS + [var + "_m" for var in VARS]
VARS = VARS + VARS_with_m
keep_vars = ["TLAT", "TLON", "time", "tarea", "HTE", "hte"] + VARS

sample_ds = xr.open_dataset(files[0])
drop_vars = [var for var in sample_ds.variables if var not in keep_vars]


time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_wim = xr.open_mfdataset(
    files,
    combine="by_coords",
    decode_times=False,
    # use_cftime=True,      # ensures all times use cftime objects
    drop_variables=drop_vars,
    # decode_times=time_coder,
)

rename_dict = {var: var[:-2] for var in ds_wim.data_vars if var.endswith("_m")}
ds_wim = ds_wim.rename(rename_dict)
ds_wim = ds_wim.roll(ni=80, roll_coords=True)

ds_wim_time = xr.Dataset(coords={"time": ds_wim["time"]})
ds_wim_time = xr.decode_cf(ds_wim_time)
time_dt = ds_wim_time.time

ds_wim["time"] = pd.to_datetime(time_dt) - time_delta
ds_wim
ds_wim['HTE'] = ds_mcw['HTE']
ds_wim

In [ ]:
# pd.to_datetime(time_dt) - time_delta

In [ ]:
# time_dt - time_delta

In [ ]:
ds_wim['fsdrad'].isel(time=-180).plot()

### Read in CICE data

In [ ]:
histfreq = "d"
experiment = "baseline-1deg"
expt_path, fig_path = get_path(experiment)


path = os.path.join(expt_path, experiment, "history/")
if histfreq == "h":
    file_pattern = os.path.join(path, "iceh_01h.????-??-??-?????.nc")
    time_delta = timedelta(minutes=30)
elif histfreq == "d":
    file_pattern = os.path.join(path, "iceh.????-??-??.nc")
    time_delta = timedelta(hours=12)
elif histfreq == "m":
    file_pattern = os.path.join(path, "iceh.????-??.nc")  # Adjust this if needed
    time_delta = timedelta(days=16)
files = sorted(glob.glob(file_pattern))

if not files:
    print(f"❌ No files found for {experiment}, skipping...")

basic_ice_vars = ["aice", "hi", "iage", "fsdrad"]
atm_forcing_vars = ["Tair", "uatm", "vatm", "Qref", "fswdn", "flwdn", "snow"]
thermo_vars = ["meltb", "meltl", "meltt"]
# 10m Air temperature, wind components, 2m specific humiditiy, incoming long wave radiation, incoming long wave radiation
# Tair, uatm, vatm, Qa, fsw, flw, fsnow from ice_forcing.F90/JRA55_data

ocn_forcing_vars = ["sst", "sss", "uocn", "vocn"]
wave_forcing_vars = ["wave_sig_ht"] # peak period? MWD?

VARS = basic_ice_vars + atm_forcing_vars + ocn_forcing_vars + wave_forcing_vars + thermo_vars
VARS_with_m = VARS + [var + "_m" for var in VARS]
VARS = VARS + VARS_with_m
keep_vars = ["TLAT", "TLON", "time", "tarea"] + VARS

sample_ds = xr.open_dataset(files[0])
drop_vars = [var for var in sample_ds.variables if var not in keep_vars]


time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds_cice = xr.open_mfdataset(
    files[:11*365],
    # combine="by_coords",
    # use_cftime=True,      # ensures all times use cftime objects
    drop_variables=drop_vars,
    # decode_times=time_coder,
    # decode_times=False,
)

rename_dict = {var: var[:-2] for var in ds_cice.data_vars if var.endswith("_m")}
ds_cice = ds_cice.rename(rename_dict)

ds_cice["time"] = ds_cice["time"].to_pandas()# - time_delta
ds_cice


In [ ]:
ds_cice = xr.open_mfdataset(
    files[:10*365],
    # combine="by_coords",
    # use_cftime=True,      # ensures all times use cftime objects
    drop_variables=drop_vars,
    # decode_times=time_coder,
    # decode_times=False,
)

In [ ]:
ds_cice["time"] = ds_cice["time"].to_pandas() #- time_delta
ds_cice


## Compare SIEs with CDR Satellite Observations
Code taken from `Single_Run_Analysis.ipynb`

In [ ]:
# files[::100]

### Read in CDR data

In [ ]:
from xarray import DataTree, map_over_datasets
OBS_TIME_SLICE = slice("1979", "2022")
sh_obs_url = "https://polarwatch.noaa.gov/erddap/griddap/nsidcG02202v4shmday"
nh_obs_url = "https://polarwatch.noaa.gov/erddap/griddap/nsidcG02202v4nhmday"


def open_cdr_dataset(path, area_file):
    ds = xr.open_dataset(path).rename(
        {'cdr_seaice_conc_monthly': 'cdr_conc', 'xgrid':'x','ygrid':'y'}
    )

    # # we also need the area of each gridcell
    areasNd = np.fromfile(area_file, dtype=np.int32).reshape(
        ds.cdr_conc.isel(time=0).shape
    )
    # # Divide by 1000 to get km2 (https://web.archive.org/web/20170817210544/http://nsidc.org/data/polar-stereo/tools_geo_pixel.html#pixel_area)
    areasKmNd_sh = areasNd / 1000
        
    ds["area"] = xr.DataArray(areasKmNd_sh, dims=["y", "x"])
    ds = ds.set_coords("area")

    ds["cdr_conc"] = ds.cdr_conc.where(ds.cdr_conc<=1)  # convert error codes to Nan

    return ds

!wget --ftp-user=anonymous -nc ftp://sidads.colorado.edu/DATASETS/seaice/polar-stereo/tools/pss25area_v3.dat ftp://sidads.colorado.edu/DATASETS/seaice/polar-stereo/tools/psn25area_v3.dat

sh_cdr_xr = open_cdr_dataset(sh_obs_url, "pss25area_v3.dat")

nh_cdr_xr = open_cdr_dataset(
    nh_obs_url,
    'psn25area_v3.dat'
)

cdr_dt = DataTree.from_dict(
    {
        "cdr_sh": sh_cdr_xr,
        'cdr_nh':nh_cdr_xr
    }
)

In [ ]:
def sea_ice_area(sic, area, range=[0.15, 1]):
    return (sic * area).where((sic >= range[0]) * (sic <= range[1])).cf.sum(["x", "y"])
    
def sea_ice_area_model(sic, area, range=[0.15, 1]):
    return (sic * area).where((sic >= range[0]) * (sic <= range[1])).sum(["ni", "nj"])

def sea_ice_extent(sic, area, range=[0.15, 1]):
    return (area).where((sic >= range[0]) * (sic <= range[1])).cf.sum(["x", "y"])

def sea_ice_extent_model(sic, area, range=[0.15, 1]):
    return (area).where((sic >= range[0]) * (sic <= range[1])).sum(["ni", "nj"])

In [ ]:
def sea_ice_area_obs(ds):

    # root dataset in datatree is not used
    if ds is None or 'time' not in ds:
        return ds
    
    sic = ds.cdr_conc
    result = sea_ice_area(sic, sic.area).to_dataset(name="cdr_area")

    # Theres a couple of data gaps which should be nan
    result.loc[{"time": "1988-01-01"}] = np.nan
    result.loc[{"time": "1987-12"}] = np.nan

    return result.sel(time=OBS_TIME_SLICE)
obs_area_dt = cdr_dt.map_over_datasets(sea_ice_area_obs)

In [ ]:
def sea_ice_extent_obs(ds):

    # root dataset in datatree is not used
    if ds is None or 'time' not in ds:
        return ds
    
    sic = ds.cdr_conc
    result = sea_ice_extent(sic, sic.area).to_dataset(name="cdr_area")

    # Theres a couple of data gaps which should be nan
    result.loc[{"time": "1988-01-01"}] = np.nan
    result.loc[{"time": "1987-12"}] = np.nan

    return result.sel(time=OBS_TIME_SLICE)

In [ ]:
obs_area_dt = sea_ice_area_obs(cdr_dt)
obs_si_area = (cdr_dt['cdr_sh']['cdr_conc'] * cdr_dt['cdr_sh']['cdr_conc'].area).sum(dim=["x", "y"]) 
obs_si_extent = (cdr_dt['cdr_sh']['cdr_conc'].area).where((cdr_dt['cdr_sh']['cdr_conc'] >= 0.15) * (cdr_dt['cdr_sh']['cdr_conc'] <= 1.0)).sum(dim=["x", "y"]) 
obs_si_extent_nh = (cdr_dt['cdr_nh']['cdr_conc'].area).where((cdr_dt['cdr_nh']['cdr_conc'] >= 0.15) * (cdr_dt['cdr_nh']['cdr_conc'] <= 1.0)).sum(dim=["x", "y"]) 

### Calculate SIE and SIA from the models

In [ ]:
ds_sia_mcw = sea_ice_area_model(ds_mcw['aice'], ds_mcw.tarea).compute()
ds_sia_mcw

In [ ]:
ds_sia_om2 = sea_ice_area_model(ds_om2['aice'], ds_mcw.tarea).compute()
ds_sia_om2

In [ ]:
ds_sia_wim = sea_ice_area_model(ds_wim['aice'], ds_wim.tarea).compute()
ds_sia_wim

In [ ]:
ds_sia_cice = sea_ice_area_model(ds_cice['aice'], ds_cice.tarea).compute()
ds_sia_cice

In [ ]:
# @map_over_subtree
def calculate_SIA_SIE_model(ds):
    # Compute for Southern Hemisphere
    sic_south = ds.aice.where(ds.TLAT < 0)
    area_south_km2 = ds.tarea.where(ds.TLAT < 0) / 1e6
    si_area_south = sea_ice_area_model(sic_south, area_south_km2).to_dataset(name="si_area_south")
    si_extent_south = sea_ice_extent_model(sic_south, area_south_km2).to_dataset(name="si_extent_south")

    # Compute for Northern Hemisphere
    sic_north = ds.aice.where(ds.TLAT > 0)
    area_north_km2 = ds.tarea.where(ds.TLAT > 0) / 1e6
    si_area_north = sea_ice_area_model(sic_north, area_north_km2).to_dataset(name="si_area_north")
    si_extent_north = sea_ice_extent_model(sic_north, area_north_km2).to_dataset(name="si_extent_north")

    # Merge both hemispheres into a single dataset
    return xr.merge([si_area_south, si_extent_south, si_area_north, si_extent_north]).load()


In [ ]:
ds_mcw['aice'] = ds_mcw.aice
mcw_SIA_SIE = calculate_SIA_SIE_model(ds_mcw)
mcw_SIA_SIE

In [ ]:
om2_SIA_SIE = calculate_SIA_SIE_model(ds_om2)
om2_SIA_SIE

In [ ]:
wim_SIA_SIE = calculate_SIA_SIE_model(ds_wim)
wim_SIA_SIE

In [ ]:
cice_SIA_SIE = calculate_SIA_SIE_model(ds_cice)
cice_SIA_SIE

In [ ]:
cice_time = pd.to_datetime(cice_SIA_SIE.time.values, unit='D', origin='2010-01-01')
cice_time

In [ ]:
# cice_SIA_SIE_monthly = cice_SIA_SIE.isel(time=slice(0,-1,30))
# cice_SIA_SIE_monthly = cice_SIA_SIE_monthly.rename({"time": "month"})
# cice_SIA_SIE_monthly
cice_SIA_SIE = cice_SIA_SIE.assign_coords(time=cice_time)

In [ ]:
sns.set_style("ticks")
plotfolder

In [ ]:
def plot_line(ax, ds, var, label, color, lw=1):
    grouped = ds[var].groupby("time.month")
    mean = grouped.mean() * 1e-6
    std = grouped.std() * 1e-6 
    mean.plot(ax=ax, label=label, color=color, lw=lw)
    ax.fill_between(mean['month'], mean - std, mean + std, alpha=0.2, color=color)

    

cmap = sns.color_palette(n_colors=5)
linewidth = 0.5

fig, axes = plt.subplots(nrows=2, figsize=(8, 6), sharex=True)

# Arctic
grouped = obs_si_extent_nh.groupby("time.month")
mean = grouped.mean() * 1e-6
std = grouped.std() * 1e-6 
mean.plot(ax=axes[0], label="CDR", color='k', lw=2)
axes[0].fill_between(mean['month'], mean - std, mean + std, alpha=0.1, color='k')

plot_line(axes[0], cice_SIA_SIE, 'si_extent_north', 'CICE6', 'tab:blue')
plot_line(axes[0], wim_SIA_SIE, 'si_extent_north', 'CICE6-WIM', 'tab:green')
plot_line(axes[0], mcw_SIA_SIE, 'si_extent_north', 'MOM6–CICE6–WW3', 'tab:red', lw=2)

# plot_line(axes[0], cice_SIA_SIE_monthly, 'si_extent_north', 'CICE6', 'tab:orange')

# Antarctic
grouped = obs_si_extent.groupby("time.month")
mean = grouped.mean() * 1e-6
std = grouped.std() * 1e-6 
mean.plot(ax=axes[1], label="CDR", color='k', lw=2)
axes[1].fill_between(mean['month'], mean - std, mean + std, alpha=0.1, color='k')

plot_line(axes[1], cice_SIA_SIE, 'si_extent_south', 'CICE6', 'tab:blue')
plot_line(axes[1], wim_SIA_SIE, 'si_extent_south', 'CICE6-WIM', 'tab:green')
plot_line(axes[1], mcw_SIA_SIE, 'si_extent_south', 'MOM6–CICE6–WW3', 'tab:red', lw=2)



# Labels
axes[0].set_ylabel("Arctic sea ice extent\n (10$^6$ km$^2$)")
axes[1].set_ylabel("Antarctic sea ice extent\n (10$^6$ km$^2$)")

axes[0].legend(frameon=False,)
axes[1].legend(frameon=False,)

for ax in axes:
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
    ax.set_xlabel('')

plt.savefig(f"{plotfolder}/SIE_climatology_ts.png", dpi=600, bbox_inches="tight") 
plt.savefig(
    f"{plotfolder}/SIE_climatology_ts_transparent.png",
    dpi=600,
    bbox_inches="tight",
    transparent=True
)
plt.show()

## Compare MIZ maps with NSIDC
Code taken from `MIZ_Comparison_With_Fraser.ipynb`

In [ ]:
# ni_idx = 150  # 200

# plt.figure(figsize=(4, 3))  # width, height in inches
# plt.plot(
#     ds_wim['TLAT'].isel(ni=ni_idx),
#     (ds_wim['HTE'] / 1e3).isel(ni=ni_idx),
# )
# plt.axhline(68, color='gray', ls='--', zorder=0)
# plt.axhline(44, color='gray', ls='--', zorder=0)

# plt.xlabel('Latitude [deg]')
# plt.ylabel('Cell width [km]')
# plt.tight_layout()

In [ ]:
time_idx = 60
ryf = False
if ryf:
    # RYF RUN
    t0 = pd.Timestamp(ds.time.values[time_idx])
    t_str = t0.strftime("%Y-%m-%d")
    day = t0.strftime("%d")
    month = t0.strftime("%m")
    year = '1990'
    if int(year) < 2010: year = '2010' # Obs start in 2010
    # day_of_year = (t0 - cftime.DatetimeNoLeap(t0.year, 1, 1)).days + 1
    day_of_year = t0.dayofyear
else:
    t0 = pd.Timestamp(ds_mcw.time.values[time_idx])
    t_str = t0.strftime("%Y-%m-%d")
    day = t0.strftime("%d")
    month = t0.strftime("%m")
    year = t0.strftime("%Y")
    day_of_year = t0.dayofyear


In [ ]:
def get_ice_charts_data(base_path, folder_name, year):
    '''
    Download MIZ extents (using the sea ice concentration defintion) from NSIDC.
    '''
    print_warn = False
    
    if "antarctic" in folder_name.lower():
        url = f"https://noaadata.apps.nsidc.org/NOAA/G10017/south/{year}/{folder_name}.zip"
    else:
        url = f"https://noaadata.apps.nsidc.org/NOAA/G10017/north/{year}/{folder_name}.zip"

    
    # Year must be a string
    year = str(year)
    
    # File paths
    zip_file = os.path.join(base_path, year, folder_name + ".zip")
    extract_dir = os.path.join(base_path, year, folder_name)

    # If extracted directory already exists, do nothing
    if os.path.isdir(extract_dir):
        if print_warn: print(f"{extract_dir} already exists — skipping download & extraction.")
        return

    year_dir = os.path.join(base_path, year)

    # Create year directory if missing
    if not os.path.isdir(year_dir):
        print(f"Creating directory for year {year}: {year_dir}")
        os.makedirs(year_dir, exist_ok=True)

    # Download the file if missing
    try:
        head = requests.head(url)
        if head.status_code != 200:
            print(f"URL does not exist ({url}) — skipping download.")
            return
    except requests.RequestException as e:
        print(f"Error checking URL: {e} — skipping download.")
        return
        
    if not os.path.exists(zip_file):
        if print_warn: print(f"{zip_file} not found. Downloading…")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(zip_file, "wb") as f:   # download directly to ZIP file
            for chunk in r.iter_content(8192):
                f.write(chunk)
    else:
        print("ZIP file already exists — skipping download.")
        

    if os.path.exists(zip_file):
        if not zipfile.is_zipfile(zip_file):
            print("WARNING: File is not a ZIP archive.")
            raise ValueError("USNIC file is not a valid ZIP.")
    
    if not os.path.exists(extract_dir):
        print(f"Extracting into {extract_dir}…")
        os.makedirs(extract_dir)
        with zipfile.ZipFile(zip_file, "r") as z:
            z.extractall(extract_dir)
         # Delete the zip file
        try:
            os.remove(zip_file)
            if print_warn: print(f"Deleted ZIP file: {zip_file}")
        except Exception as e:
            print(f"WARNING: Could not delete ZIP file: {e}")
    else:
        if print_warn: print(f"{extract_dir} already exists — skipping extraction.")
    return 


In [ ]:
def get_proj(hemisphere):
    if hemisphere.lower().startswith('s'):
        projection = ccrs.SouthPolarStereo(central_longitude=0)
        extent = [-180, 180, -90, -40]
    elif hemisphere.lower().startswith('n'):
        projection = ccrs.Stereographic(
            central_latitude=90.0,
            central_longitude=-45.0,
            true_scale_latitude=70.0,
            globe=ccrs.Globe(semimajor_axis=6378273.0, semiminor_axis=6356889.448910593)
        )
        extent = [0, 360, 40, 90]
    else:
        raise ValueError("hemisphere must be 'north' or 'south'")
    return projection, extent

Northern hemisphere

In [ ]:
folder_name = f"nic_miz{year}{day_of_year:03d}nc_pl_a"
base_path = "/g/data/ps29/nd0349/datasets/USNIC/ice_charts/arctic/"
get_ice_charts_data(base_path, folder_name, year)

In [ ]:
# Plot aesthetics
color_miz = "tab:red"
color_ice = "tab:blue"
bounds = [0.0, 0.1, 0.8, 1.0]
cmap = ListedColormap(["white", color_miz, color_ice])  
norm = mcolors.BoundaryNorm(bounds, cmap.N)


# Make figure
number_panels = 4
hemisphere = "north"
projection, extent = get_proj(hemisphere)
    
fig, axes = plt.subplots(
    ncols=number_panels,
    subplot_kw={'projection': projection},
    figsize=(4 * number_panels, 4),
    gridspec_kw={'wspace': 0.25, 'hspace': 0.15}
)

for ax in axes:
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                                    edgecolor='none',
                                    facecolor='gray', linewidth=0.5)
    ax.coastlines(resolution='50m')
    ax.add_feature(land_50m)
    # Make a circle plot
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.set_title("NO DATA")


# Load shapefile
cmap_obs = ListedColormap([color_miz, color_ice])  
bounds_obs = [0, 1, 2]
norm_obs = BoundaryNorm(bounds_obs, cmap_obs.N)

shp_file = glob.glob(os.path.join(base_path, year, folder_name, "*.shp"))[0]
gdf = gpd.read_file(shp_file)

# Reproject
gdf_proj = gdf.to_crs(epsg=3413)

gdf_proj.plot(
    ax=axes[0],
    column="ICECODE",
    cmap=cmap_obs,
    norm=norm_obs,
    edgecolor="black",
    linewidth=0.5,
    legend=True, 
    figsize=(8, 8),
)

leg = axes[0].get_legend()
for text, new_label in zip(leg.get_texts(), ['SIC 10–80%', 'SIC >80%']):
    text.set_text(new_label)
leg.set_loc('upper center')
leg.set_bbox_to_anchor((1.125, 1.1))
axes[0].set_title(f"{year}-{month}-{day}")

ref_doy = t0.timetuple().tm_yday
# ds_mcw.sel(time=t0)['aice'].plot.contourf(
ds_mcw_plot = ds_mcw.sel(time=ds_mcw.time.dt.dayofyear == ref_doy)['aice'].isel(time=0)
ds_mcw_plot["TLON"].values = ds_wim["TLON"].values
ds_mcw_plot["TLAT"].values = ds_wim["TLAT"].values
ds_mcw_plot.plot.pcolormesh(
    ax=axes[1],
    x="TLON",
    y="TLAT",
    levels=bounds,  
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)

# cs = ds_mcw_plot.plot.contour(
#     ax=axes[1],
#     x="TLON",
#     y="TLAT",
#     levels=bounds[1:],  
#     colors='k',    
#     linewidths=0.5,
#     transform=ccrs.PlateCarree()
# )

axes[1].set_title(t_str)


# CICE6-WIM
t_gregorian = cftime.DatetimeGregorian(
    t0.year,
    t0.month,
    t0.day,
    t0.hour,
    t0.minute,
    t0.second,
)
ref_doy = t_gregorian.timetuple().tm_yday
ref_doy

ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.pcolormesh(
    ax=axes[2],
    x="TLON",
    y="TLAT",
    levels=bounds,  
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)

# cs = ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.contour(
#     ax=axes[2],
#     x="TLON",
#     y="TLAT",
#     levels=bounds[1:],  
#     colors='k',    
#     linewidths=0.5,
#     transform=ccrs.PlateCarree()
# )
t = ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy).time.values.item()

axes[2].set_title(t.strftime("%Y-%m-%d"))


# CICE6
t_gregorian = cftime.DatetimeGregorian(
    t0.year,
    t0.month,
    t0.day,
    t0.hour,
    t0.minute,
    t0.second,
)
ref_doy = t_gregorian.timetuple().tm_yday
ref_doy

ds_cice.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.pcolormesh(
    ax=axes[3],
    x="TLON",
    y="TLAT",
    levels=bounds,  
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)

# cs = ds_cice.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.contour(
#     ax=axes[2],
#     x="TLON",
#     y="TLAT",
#     levels=bounds[1:],  
#     colors='k',    
#     linewidths=0.5,
#     transform=ccrs.PlateCarree()
# )
t = ds_cice.sel(time=ds_cice.time.dt.dayofyear == ref_doy).time.values.item()

axes[3].set_title(t.strftime("%Y-%m-%d"))


plt.savefig(f"{plotfolder}/Arctic_MIZ_SIC_maps_{ref_doy}.png", dpi=600, bbox_inches="tight") 

Southern hemisphere

In [ ]:
time_idx = 180
ryf = False
if ryf:
    # RYF RUN
    t0 = pd.Timestamp(ds.time.values[time_idx])
    t_str = t0.strftime("%Y-%m-%d")
    day = t0.strftime("%d")
    month = t0.strftime("%m")
    year = '1990'
    if int(year) < 2010: year = '2010' # Obs start in 2010
    # day_of_year = (t0 - cftime.DatetimeNoLeap(t0.year, 1, 1)).days + 1
    day_of_year = t0.dayofyear
else:
    t0 = pd.Timestamp(ds_mcw.time.values[time_idx])
    t_str = t0.strftime("%Y-%m-%d")
    day = t0.strftime("%d")
    month = t0.strftime("%m")
    year = t0.strftime("%Y")
    day_of_year = t0.dayofyear


In [ ]:

folder_name = f"nic_miz{year}{day_of_year:03d}sc_pl_a"
base_path = "/g/data/ps29/nd0349/datasets/USNIC/ice_charts/antarctic/"
get_ice_charts_data(base_path, folder_name, year)

In [ ]:
# Plot aesthetics
color_miz = "tab:red"
color_ice = "tab:blue"
bounds = [0.0, 0.1, 0.8, 1.0]
cmap = ListedColormap(["white", color_miz, color_ice])  
norm = mcolors.BoundaryNorm(bounds, cmap.N)


# Make figure
number_panels = 3
hemisphere = "south"
projection, extent = get_proj(hemisphere)
    
fig, axes = plt.subplots(
    ncols=number_panels,
    subplot_kw={'projection': projection},
    figsize=(4 * number_panels, 4),
    gridspec_kw={'wspace': 0.25, 'hspace': 0.15}
)

for ax in axes:
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                                    edgecolor='none',
                                    facecolor='gray', linewidth=0.5)
    ax.coastlines(resolution='50m')
    ax.add_feature(land_50m)
    # Make a circle plot
    theta = np.linspace(0, 2*np.pi, 100)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.set_title("NO DATA")


# Load shapefile
cmap_obs = ListedColormap([color_miz, color_ice])  
bounds_obs = [0, 1, 2]
norm_obs = BoundaryNorm(bounds_obs, cmap_obs.N)

shp_file = glob.glob(os.path.join(base_path, year, folder_name, "*.shp"))[0]
gdf = gpd.read_file(shp_file)

# Reproject
gdf_proj = gdf.to_crs(epsg=3031)

gdf_proj.plot(
    ax=axes[0],
    column="ICECODE",
    cmap=cmap_obs,
    norm=norm_obs,
    edgecolor="black",
    linewidth=0.5,
    legend=True, 
    figsize=(8, 8),
    # transform=ccrs.Projection()
)

leg = axes[0].get_legend()
for text, new_label in zip(leg.get_texts(), ['SIC 10–80%', 'SIC >80%']):
    text.set_text(new_label)
leg.set_loc('upper center')
leg.set_bbox_to_anchor((1.125, 1.1))
axes[0].set_title(f"{year}-{month}-{day}")

ds_mcw.sel(time=t0)['aice'].plot.contourf(
    ax=axes[1],
    x="TLON",
    y="TLAT",
    levels=bounds,  
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)

cs = ds_mcw.sel(time=t0)['aice'].plot.contour(
    ax=axes[1],
    x="TLON",
    y="TLAT",
    levels=bounds[1:],  
    colors='k',    
    linewidths=0.5,
    transform=ccrs.PlateCarree()
)

axes[1].set_title(t_str)


# CICE6-WIM
t_gregorian = cftime.DatetimeGregorian(
    t0.year,
    t0.month,
    t0.day,
    t0.hour,
    t0.minute,
    t0.second,
)
ref_doy = t_gregorian.timetuple().tm_yday
ref_doy

ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.pcolormesh(
    ax=axes[2],
    x="TLON",
    y="TLAT",
    levels=bounds,  
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    add_colorbar=False
)

cs = ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy)['aice'].isel(time=0).plot.contour(
    ax=axes[2],
    x="TLON",
    y="TLAT",
    levels=bounds[1:],  
    colors='k',    
    linewidths=0.5,
    transform=ccrs.PlateCarree()
)
t = ds_wim.sel(time=ds_wim.time.dt.dayofyear == ref_doy).time.values.item()

axes[2].set_title(t.strftime("%Y-%m-%d"))

plt.savefig(f"{plotfolder}/Antarctic_MIZ_SIC_maps_{ref_doy}.png", dpi=600, bbox_inches="tight") 

### SIE time series

In [ ]:
# Take the 15th of each month
days_of_year = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
years = np.arange(2010, 2026)

base_path = "/g/data/ps29/nd0349/datasets/USNIC/ice_charts/antarctic/"

rows = []   # List of dictionaries for final dataframe

for year in tqdm(years):
    for i, day_of_year in enumerate(days_of_year):
        folder_name = f"nic_miz{year}{day_of_year:03d}sc_pl_a"
        get_ice_charts_data(base_path, folder_name, year)
        shp_file = glob.glob(os.path.join(base_path, str(year), folder_name, "*.shp"))

        # Default values
        miz_extent = np.nan
        inner_extent = np.nan

        if not shp_file:
            print(f"No shapefile for {folder_name} — skipping.")
        else:
            gdf = gpd.read_file(shp_file[0])

            # Find correct area column
            possible_cols = ["SHAPE_Area", "Shape_Area"]
            area_col = next((c for c in possible_cols if c in gdf.columns), None)

            if area_col is None:
                print(f"No area column for {folder_name}. Columns: {list(gdf.columns)}")
                print(gdf)
            else:
                areas = gdf.groupby("ICECODE")[area_col].sum()
                miz_extent = areas.iloc[0]
                inner_extent = areas.iloc[1]

        month = i + 1
        rows.append({
            "year": year,
            "month": month,
            "date": datetime(year, month, 15),
            "miz_extent": miz_extent,
            "inner_extent": inner_extent,
            "sea_ice_extent": miz_extent + inner_extent
        })

df = pd.DataFrame(rows)
df

Calculate climatology of observations

In [ ]:
df_clim_antarctic = df.groupby('month').mean()
del df
df_clim_antarctic

In [ ]:
# Take the 15th of each month
days_of_year = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
years = np.arange(2010, 2026)

base_path = "/g/data/ps29/nd0349/datasets/USNIC/ice_charts/arctic/"

rows = []   # List of dictionaries for final dataframe

for year in tqdm(years):
    for i, day_of_year in enumerate(days_of_year):
        folder_name = f"nic_miz{year}{day_of_year:03d}nc_pl_a"
        get_ice_charts_data(base_path, folder_name, year)
        shp_file = glob.glob(os.path.join(base_path, str(year), folder_name, "*.shp"))

        # Default values
        miz_extent = np.nan
        inner_extent = np.nan

        if not shp_file:
            print(f"No shapefile for {folder_name} — skipping.")
        else:
            gdf = gpd.read_file(shp_file[0])

            # Find correct area column
            possible_cols = ["SHAPE_Area", "Shape_Area"]
            area_col = next((c for c in possible_cols if c in gdf.columns), None)

            if area_col is None:
                print(f"No area column for {folder_name}. Columns: {list(gdf.columns)}")
                print(gdf)
            else:
                areas = gdf.groupby("ICECODE")[area_col].sum()
                miz_extent = areas.iloc[0]
                inner_extent = areas.iloc[1]

        month = i + 1
        rows.append({
            "year": year,
            "month": month,
            "date": datetime(year, month, 15),
            "miz_extent": miz_extent,
            "inner_extent": inner_extent,
            "sea_ice_extent": miz_extent + inner_extent
        })

df = pd.DataFrame(rows)
df

In [ ]:
df_clim_arctic = df.groupby('month').mean()
del df
df_clim_arctic

### Calculate the model's climatology

In [ ]:
ds_mcw_clim = ds_mcw.groupby('time.month').mean('time').compute()
ds_mcw_clim

In [ ]:
ds_om2_clim = ds_om2.groupby('time.month').mean('time').compute()
ds_om2_clim

In [ ]:
ds_wim_clim = ds_wim.groupby('time.month').mean('time').compute()
ds_wim_clim

### Compare MIZ extents against NSIDC

In [ ]:
def _get_mask(ds, definition, hemisphere, threshold=None):
    # Set defaults for each definition
    defaults = {'wave_sig_ht': 0.05, 'aice': 0.8, 'fsdrad': 200}
    if threshold is None:
        try:
            threshold = defaults[definition]
        except KeyError:
            raise ValueError(f"Unknown definition: {definition}")

    # Example for aice
    aice_var = _get_var(ds, ['aice_m', 'aice'])
    wave_var = _get_var(ds, ['wave_sig_ht_m', 'wave_sig_ht'])
    fsd_var = _get_var(ds, ['fsdrad_m', 'fsdrad'])
    
    # Hemisphere mask
    if hemisphere == 'south':
        hemi_mask = ds['TLAT'] < 0
    elif hemisphere == 'north':
        hemi_mask = ds['TLAT'] > 0
    else:
        raise ValueError("hemisphere must be 'north' or 'south'")

    # Base ice presence mask
    base_mask = ds[aice_var] > 0.15

    # Definition-specific condition
    if definition == 'wave_sig_ht':
        cond = ds[wave_var] > threshold
    elif definition == 'aice':
        cond = ds[aice_var] < threshold
    elif definition == 'fsdrad':
        cond = ds[fsd_var] < threshold
    elif definition in ['SIE', 'SIA']:
        cond = base_mask
    else:
        raise ValueError(f"Unknown definition: {definition}")

    return base_mask & cond & hemi_mask

def _get_var(ds, names):
    """Return the first matching variable name in ds."""
    for n in names:
        if n in ds:
            return n
    raise KeyError(f"None of the variables {names} found in dataset.")

def _integrate_area(ds, mask, definition):
    if definition == "SIA":
        return ds['tarea']*ds['aice'].where(mask).sum(dim=['ni', 'nj'])
    else:
        return ds['tarea'].where(mask).sum(dim=['ni', 'nj'])

def calculate_area(ds_cice, definition='wave', threshold=0.05, integration='simple', 
                        measure='median', mask=True, method='naive', hemisphere='south'):
    mask_2d = _get_mask(ds_cice, definition, hemisphere, threshold)
    
    sea_ice_extent = _integrate_area(ds_cice, mask_2d, definition) * 1e-12

    if definition in ["wave_sig_ht", "aice", "fsdrad"]:
        name = "MIZ area"
    elif definition == "SIE":
        name = "Sea ice extent"
    elif definition == "SIA":
        name = "Sea ice area"

    return xr.DataArray(
        sea_ice_extent,
        dims=sea_ice_extent.dims,
        coords={'month': ds_cice['month']},
        name=name,
        attrs={
            'units': '10^6 km^2',
            'description': f'MIZ extent ({definition})'
        }
    )


In [ ]:
# Calculate Antarctic extents
hemisphere = "south"
mcw_antarctic_sie = calculate_area(ds_mcw_clim, definition='SIE', hemisphere=hemisphere)
mcw_antarctic_sia = calculate_area(ds_mcw_clim, definition='SIA', hemisphere=hemisphere)
mcw_antarctic_miz_extent_wave = calculate_area(ds_mcw_clim, definition='wave_sig_ht', hemisphere=hemisphere)
mcw_antarctic_miz_extent_aice = calculate_area(ds_mcw_clim, definition='aice', threshold=0.8, hemisphere=hemisphere)
mcw_antarctic_miz_extent_fsdrad = calculate_area(ds_mcw_clim, definition='fsdrad', threshold=200, hemisphere=hemisphere)

In [ ]:
wim_antarctic_sie = calculate_area(ds_wim_clim, definition='SIE', hemisphere=hemisphere)
wim_antarctic_sia = calculate_area(ds_wim_clim, definition='SIA', hemisphere=hemisphere)
wim_antarctic_miz_extent_wave = calculate_area(ds_wim_clim, definition='wave_sig_ht', hemisphere=hemisphere)
wim_antarctic_miz_extent_aice = calculate_area(ds_wim_clim, definition='aice', threshold=0.8, hemisphere=hemisphere)
wim_antarctic_miz_extent_fsdrad = calculate_area(ds_wim_clim, definition='fsdrad', threshold=200, hemisphere=hemisphere)

In [ ]:
# Calculate Arctic extents
hemisphere = "north"
mcw_arctic_sie = calculate_area(ds_mcw_clim, definition='SIE', hemisphere=hemisphere)
mcw_arctic_sia = calculate_area(ds_mcw_clim, definition='SIA', hemisphere=hemisphere)
mcw_arctic_miz_extent_wave = calculate_area(ds_mcw_clim, definition='wave_sig_ht', hemisphere=hemisphere)
mcw_arctic_miz_extent_aice = calculate_area(ds_mcw_clim, definition='aice', threshold=0.8, hemisphere=hemisphere)
mcw_arctic_miz_extent_fsdrad = calculate_area(ds_mcw_clim, definition='fsdrad', threshold=200, hemisphere=hemisphere)

In [ ]:
wim_arctic_sie = calculate_area(ds_wim_clim, definition='SIE', hemisphere=hemisphere)
wim_arctic_sia = calculate_area(ds_wim_clim, definition='SIA', hemisphere=hemisphere)
wim_arctic_miz_extent_wave = calculate_area(ds_wim_clim, definition='wave_sig_ht', hemisphere=hemisphere)
wim_arctic_miz_extent_aice = calculate_area(ds_wim_clim, definition='aice', threshold=0.8, hemisphere=hemisphere)
wim_arctic_miz_extent_fsdrad = calculate_area(ds_wim_clim, definition='fsdrad', threshold=200, hemisphere=hemisphere)

### Compare MCW with NSIDC using the sea ice concentration definition

In [ ]:
color_miz = 'tab:red'
color_sie = 'tab:blue'

fig, axes = plt.subplots(
    nrows=2,
    figsize=(12, 6),
    sharex=True,
)


# SIE
ax = axes[0]
ax.plot(df_clim_arctic.index, df_clim_arctic['sea_ice_extent'] * 1e-12,
        color=color_sie,
        linestyle='-',
        label="NSIDC SIE",
)

ax.plot(mcw_arctic_miz_extent_aice.month, mcw_arctic_sie,
        color=color_sie,
        linestyle='--',
        label="MCW",
)

ax.plot(wim_arctic_miz_extent_aice.month, wim_arctic_sie,
        color=color_sie,
        linestyle=':',
        label="CICE6-WIM",
)


# MIZ 
ax.plot(df_clim_antarctic.index, df_clim_arctic['miz_extent'] * 1e-12,
        color=color_miz,
        linestyle='-',
        label="NSIDC MIZ",
)

ax.plot(mcw_arctic_miz_extent_aice.month, mcw_arctic_miz_extent_aice,
        color=color_miz,
        linestyle='--',
        label="MCW",
)

ax.plot(wim_arctic_miz_extent_aice.month, wim_arctic_miz_extent_aice,
        color=color_miz,
        linestyle=':',
        label="CICE6-WIM",
)


# SIE
ax = axes[1]
ax.plot(df_clim_antarctic.index, df_clim_antarctic['sea_ice_extent'] * 1e-12,
        color=color_sie,
        linestyle='-',
        label="NSIDC SIE",
)

ax.plot(mcw_antarctic_miz_extent_aice.month, mcw_antarctic_sie,
        color=color_sie,
        linestyle='--',
        label="MCW",
)

ax.plot(wim_antarctic_miz_extent_aice.month, wim_antarctic_sie,
        color=color_sie,
        linestyle=':',
        label="CICE6-WIM",
)


# MIZ 
ax.plot(df_clim_antarctic.index, df_clim_antarctic['miz_extent'] * 1e-12,
        color=color_miz,
        linestyle='-',
        label="NSIDC MIZ",
)

ax.plot(mcw_antarctic_miz_extent_aice.month, mcw_antarctic_miz_extent_aice,
        color=color_miz,
        linestyle='--',
        label="MCW",
)

ax.plot(wim_antarctic_miz_extent_aice.month, wim_antarctic_miz_extent_aice,
        color=color_miz,
        linestyle=':',
        label="CICE6-WIM",
)

ax.legend(loc='upper left')
ax.set_xlabel("Month")
axes[0].set_ylabel("Sea Ice Extent\n [10$^6$ km$^2$]")
axes[1].set_ylabel("Sea Ice Extent\n [10$^6$ km$^2$]")

plt.savefig(f"{plotfolder}/MIZ_SIC_SIE_ts.png", dpi=600, bbox_inches="tight") 

### Time series for one year

In [ ]:
if calendar.isleap(int(year)):
    days_of_year = np.arange(1, 367, 1)
else:
    days_of_year = np.arange(1, 366, 1)
    
base_path = "/g/data/ps29/nd0349/datasets/USNIC/ice_charts/antarctic/"

year = '2011'
ice_charts_miz = []
ice_charts_inner = []
for day_of_year in tqdm(days_of_year):
    # print(day_of_year)
    folder_name = f"nic_miz{year}{day_of_year:03d}sc_pl_a"
    get_ice_charts_data(base_path, folder_name, year)
    shp_file = glob.glob(os.path.join(base_path, year, folder_name, "*.shp"))
    
    try:
        gdf = gpd.read_file(shp_file[0])
        areas = gdf.groupby("ICECODE")["SHAPE_Area"].sum()
        ice_charts_miz.append(areas.iloc[0])
        ice_charts_inner.append(areas.iloc[1])
    except Exception as e:
        print(f"Error reading shapefile for day {day_of_year}: {e} — skipping.")
        ice_charts_miz.append(np.nan)    
        ice_charts_inner.append(np.nan)
        continue
    # gdf = gpd.read_file(shp_file)
    # areas = gdf.groupby("ICECODE")["SHAPE_Area"].sum()
    # # print(area_by_code)
    # ice_charts_miz.append(areas[0])
    # ice_charts_inner.append(areas[1])
ice_charts_miz = np.array(ice_charts_miz)
ice_charts_inner = np.array(ice_charts_inner)

## Compare MIZ widths against Alex's observations

### Calculate MIZ widths

In [ ]:
def _integrate_width(width, ds, method):
    if method == 'simple':
        return width.sum(dim='nj', skipna=True)
    elif method == 'weighted':
        aice_var = _get_var(ds, ['aice_m', 'aice'])
        return (width * ds[aice_var]).sum(dim='nj', skipna=True)
    else:
        raise ValueError(f"Unknown integration: {method}")

def _reduce_width(width, method, mask):
    if mask:
        width = width.where(width != 0)
    if method == 'median':
        return width.median(dim='ni', skipna=True)
    elif method == 'mean':
        return width.mean(dim='ni', skipna=True)
    else:
        raise ValueError(f"Unknown measure: {method}")

def calculate_miz_width(ds_cice, definition='wave', threshold=0.3, integration='simple', 
                        measure='median', mask=True, hemisphere='south', freq='monthly'):
    mask_2d = _get_mask(ds_cice, definition, hemisphere, threshold)
    
    miz_width_2d = ds_cice['HTE'].where(mask_2d) / 1000 # m to km
    miz_width_1d = _integrate_width(miz_width_2d, ds_cice, integration)
    miz_width_reduced = _reduce_width(miz_width_1d, measure, mask)

    if freq == 'monthly':
        coords = {'month': ds_cice['month']}
    elif freq == 'daily':
        coords = {'time': ds_cice['time']}
    elif freq == 'dayofyear':
        coords = {'dayofyear': ds_cice['dayofyear']}

    return xr.DataArray(
        miz_width_reduced,
        dims=miz_width_reduced.dims,
        coords=coords,
        name='miz_width',
        attrs={
            'units': 'km',
            'description': f'MIZ width ({definition})'
        }
    )

In [ ]:
# MIZ defintion thresholds
swh_threshold = 0.3
sic_threshold = 0.8
fsd_threshold = 100

In [ ]:
hemisphere='south'
mcw_antarctic_miz_width_wave = calculate_miz_width(ds_mcw_clim, definition='wave_sig_ht', hemisphere=hemisphere)
mcw_antarctic_miz_width_aice = calculate_miz_width(ds_mcw_clim, definition='aice', threshold=sic_threshold, hemisphere=hemisphere)
mcw_antarctic_miz_width_fsdrad = calculate_miz_width(ds_mcw_clim, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere)

In [ ]:
wim_antarctic_miz_width_wave = calculate_miz_width(ds_wim_clim, definition='wave_sig_ht', hemisphere=hemisphere)
wim_antarctic_miz_width_aice = calculate_miz_width(ds_wim_clim, definition='aice', threshold=sic_threshold, hemisphere=hemisphere)
wim_antarctic_miz_width_fsdrad = calculate_miz_width(ds_wim_clim, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere)

In [ ]:
hemisphere='north'
mcw_arctic_miz_width_wave = calculate_miz_width(ds_mcw_clim, definition='wave_sig_ht', hemisphere=hemisphere)
mcw_arctic_miz_width_aice = calculate_miz_width(ds_mcw_clim, definition='aice', threshold=sic_threshold, hemisphere=hemisphere)
mcw_arctic_miz_width_fsdrad = calculate_miz_width(ds_mcw_clim, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere)

In [ ]:
wim_arctic_miz_width_wave = calculate_miz_width(ds_wim_clim, definition='wave_sig_ht', hemisphere=hemisphere)
wim_arctic_miz_width_aice = calculate_miz_width(ds_wim_clim, definition='aice', threshold=sic_threshold, hemisphere=hemisphere)
wim_arctic_miz_width_fsdrad = calculate_miz_width(ds_wim_clim, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere)

### Load in observations (*Fraser et al., 2025*)

In [ ]:
def ReadInAltika(version, year=2019):
    if version == '0.6':
        month_range = range(1,13)
        df = pd.concat((pd.read_csv('/home/566/nd0349/Fraser-2024/data/v0_6/' + str(year) + ("%02d" % (month,)) + '_output_v0_6.csv') 
                        for month in tqdm(month_range, total = len(month_range), desc = "Reading in Alex's data")),
                        ignore_index=True)
        
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    
    elif version == '0.10':
        # Version 0.10
        df_raw = pd.read_csv('data/v0_10/' + str(year) + '_all_output_v0_10.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.11':
        # Version 0.11
        df_raw = pd.read_csv('data/v0_11/' + str(year) + '_all_output_v0_11.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.12':
        # Version 0.12
        df_raw = pd.read_csv('data/v0_12/' + str(year) + '_all_output_v0_12.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    elif version == '0.15':
        # Version 0.15
        df_raw = pd.read_csv('/home/566/nd0349/Fraser-2024/data/v0_15/' + str(year) + '_all_output_v0_15.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        
    # Add dates to dataframe
    df['date'] = pd.to_datetime(df["first_meas_time"])#, format='%Y-%m-%d').dt.round("d")
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    return df

In [ ]:
from tqdm.notebook import tqdm

years = range(2013, 2024)
dfs = []

# swh_min = 10**-12
# swh_max = 100
# miz_max = 10000
# miz_min = -10**-12

for year in tqdm(years):
    # print(year)
    df = pd.DataFrame(ReadInAltika(version='0.15', year=year))
    df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    df['date'] = pd.to_datetime(df["first_meas_time"]).dt.date # , format='%Y-%m-%d %H:%M:%S.%f').dt.date
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    df['mizwidth_lat'] = abs(df['latAtAltiKaEdge'] - df['latAtInnerMIZ'])*111.32 # Alex's conversion from latitudes to km

    dfs.append(df)
# Combine into one dataframe
df_fraser_all = pd.concat(dfs, ignore_index=True)
# df_tmp = df_all.copy()
# condition_met_df = (df_tmp['swhAtMyEdge'] < swh_max) & (df_tmp['swhAtMyEdge'] > swh_min) & (df_tmp['mizWidthAlongTrackFromAltikaEdge'] < miz_max) & (df_tmp['mizWidthAlongTrackFromAltikaEdge'] > miz_min)
# df_all = df_tmp.where(condition_met_df)
df_fraser_all.head()

In [ ]:
# Average monthly
df_fraser_clim = df_fraser_all.groupby('month').mean(numeric_only=True)
df_fraser_clim.head()

### Plot the different MIZ width defintions

In [ ]:
def miz_ts_figure(region=None):
    plt.rcParams['font.size'] = 12
    fig, axes = plt.subplots(ncols=2, figsize=(14,4), sharex=True, sharey=True)
    plt.subplots_adjust(wspace=0.12)

    axes[0].text(0.0, 1.03, 'a) ANTARCTIC', fontsize=12, fontweight='bold', transform=axes[0].transAxes)
    axes[1].text(0.0, 1.03, 'b) ARCTIC', fontsize=12, fontweight='bold', transform=axes[1].transAxes)
    # axes[2].text(0.0, 1.03, 'c) ACCESS-OM3 - ACCESS-OM2', fontsize=12, fontweight='bold', transform=axes[2].transAxes)

    # axes[0].text(0.99, 1.03, "ANTARCTIC", ha='right', fontsize=12, transform=axes[0].transAxes)
    # axes[1].text(0.99, 1.03, "ARCTIC", ha='right', fontsize=12, transform=axes[1].transAxes)
    
    for i, ax in enumerate(axes):
        ax.plot(np.arange(1,13,1), np.full(12,0), lw=0.5, c='k')
        ax.set_xlim([1,12])
        ax.set_xticks(np.arange(1,13,1))
        ax.set_xlabel('Month')
    
    # axes[0].set_ylabel('Sea ice volume tendency\n(x1000 km$^3$ day$^{-1}$)')
        
    return fig, axes
df_fraser_clim

In [ ]:
color_swh = 'tab:blue'
color_sic = 'lightblue'
color_fsd = 'tab:green'

# region = 'ANTARCTICA'
fig, axes = miz_ts_figure()

axes[0].plot(mcw_antarctic_miz_width_wave.month, mcw_antarctic_miz_width_wave, label='Sig. wave height', color=color_swh)
axes[0].plot(mcw_antarctic_miz_width_aice.month, mcw_antarctic_miz_width_aice, label='Ice concentration', color=color_sic)
axes[0].plot(mcw_antarctic_miz_width_fsdrad.month, mcw_antarctic_miz_width_fsdrad, label='Floe size', color=color_fsd)

axes[0].plot(wim_antarctic_miz_width_wave.month, wim_antarctic_miz_width_wave, color=color_swh, linestyle='--')
axes[0].plot(wim_antarctic_miz_width_aice.month, wim_antarctic_miz_width_aice, color=color_sic, linestyle='--')
axes[0].plot(wim_antarctic_miz_width_fsdrad.month, wim_antarctic_miz_width_fsdrad, color=color_fsd, linestyle='--')

axes[0].plot(df_fraser_clim.index, df_fraser_clim['mizWidthAlongTrackFromMyEdge'], color='k', label="Observations")

axes[0].legend(frameon=False, ncols=1, fontsize=12)

axes[1].plot(mcw_arctic_miz_width_wave.month, mcw_arctic_miz_width_wave, label='Sig. wave height', color=color_swh)
axes[1].plot(mcw_arctic_miz_width_aice.month, mcw_arctic_miz_width_aice, label='Ice concentration', color=color_sic)
axes[1].plot(mcw_arctic_miz_width_fsdrad.month, mcw_arctic_miz_width_fsdrad, label='Floe size', color=color_fsd)

axes[1].plot(wim_arctic_miz_width_wave.month, wim_arctic_miz_width_wave, color=color_swh, linestyle='--')
axes[1].plot(wim_arctic_miz_width_aice.month, wim_arctic_miz_width_aice, color=color_sic, linestyle='--')
axes[1].plot(wim_arctic_miz_width_fsdrad.month, wim_arctic_miz_width_fsdrad, color=color_fsd, linestyle='--')

axes[0].set_ylabel("MIZ width [km]")

plt.savefig(f"{plotfolder}/MIZ_width_all_definitions_ts.png", dpi=600, bbox_inches="tight") 

### MIZ area

In [ ]:
fig, axes = miz_ts_figure()

axes[0].plot(mcw_antarctic_miz_extent_wave.month, mcw_antarctic_miz_extent_wave, label='Sig. wave height', color=color_swh)
axes[0].plot(mcw_antarctic_miz_extent_aice.month, mcw_antarctic_miz_extent_aice, label='Ice concentration', color=color_sic)
axes[0].plot(mcw_antarctic_miz_extent_fsdrad.month, mcw_antarctic_miz_extent_fsdrad, label='Floe size', color=color_fsd)

axes[0].plot(wim_antarctic_miz_extent_wave.month, wim_antarctic_miz_extent_wave, color=color_swh, linestyle='--')
axes[0].plot(wim_antarctic_miz_extent_aice.month, wim_antarctic_miz_extent_aice, color=color_sic, linestyle='--')
axes[0].plot(wim_antarctic_miz_extent_fsdrad.month, wim_antarctic_miz_extent_fsdrad, color=color_fsd, linestyle='--')

axes[0].legend(frameon=False, ncols=1, fontsize=12)

axes[1].plot(mcw_arctic_miz_extent_wave.month, mcw_arctic_miz_extent_wave, label='Sig. wave height', color=color_swh)
axes[1].plot(mcw_arctic_miz_extent_aice.month, mcw_arctic_miz_extent_aice, label='Ice concentration', color=color_sic)
axes[1].plot(mcw_arctic_miz_extent_fsdrad.month, mcw_arctic_miz_extent_fsdrad, label='Floe size', color=color_fsd)

axes[1].plot(wim_arctic_miz_extent_wave.month, wim_arctic_miz_extent_wave, color=color_swh, linestyle='--')
axes[1].plot(wim_arctic_miz_extent_aice.month, wim_arctic_miz_extent_aice, color=color_sic, linestyle='--')
axes[1].plot(wim_arctic_miz_extent_fsdrad.month, wim_arctic_miz_extent_fsdrad, color=color_fsd, linestyle='--')

axes[0].set_ylabel("Sea Ice Area [$10^6$ km$^2$]")

plt.savefig(f"{plotfolder}/MIZ_area_all_definitions_ts.png", dpi=600, bbox_inches="tight") 

### MIZ fraction

In [ ]:
fig, axes = miz_ts_figure()

x = np.arange(1, 13)
axes[0].plot(mcw_antarctic_miz_extent_wave.month, mcw_antarctic_miz_extent_wave/mcw_antarctic_sie, label='Sig. wave height', color=color_swh)
axes[0].plot(mcw_antarctic_miz_extent_aice.month, mcw_antarctic_miz_extent_aice/mcw_antarctic_sie, label='Ice concentration', color=color_sic)
axes[0].plot(mcw_antarctic_miz_extent_fsdrad.month, mcw_antarctic_miz_extent_fsdrad/mcw_antarctic_sie, label='Floe size', color=color_fsd)

axes[0].plot(wim_antarctic_miz_extent_wave.month, wim_antarctic_miz_extent_wave/wim_antarctic_sie, color=color_swh, linestyle='--')
axes[0].plot(wim_antarctic_miz_extent_aice.month, wim_antarctic_miz_extent_aice/wim_antarctic_sie, color=color_sic, linestyle='--')
axes[0].plot(wim_antarctic_miz_extent_fsdrad.month, wim_antarctic_miz_extent_fsdrad/wim_antarctic_sie, color=color_fsd, linestyle='--')

axes[0].legend(frameon=False, ncols=1, fontsize=12, loc="lower left")

axes[1].plot(mcw_arctic_miz_extent_wave.month, mcw_arctic_miz_extent_wave/mcw_arctic_sie, label='Sig. wave height', color=color_swh)
axes[1].plot(mcw_arctic_miz_extent_aice.month, mcw_arctic_miz_extent_aice/mcw_arctic_sie, label='Ice concentration', color=color_sic)
axes[1].plot(mcw_arctic_miz_extent_fsdrad.month, mcw_arctic_miz_extent_fsdrad/mcw_arctic_sie, label='Floe size', color=color_fsd)

axes[1].plot(wim_arctic_miz_extent_wave.month, wim_arctic_miz_extent_wave/wim_arctic_sie, color=color_swh, linestyle='--')
axes[1].plot(wim_arctic_miz_extent_aice.month, wim_arctic_miz_extent_aice/wim_arctic_sie, color=color_sic, linestyle='--')
axes[1].plot(wim_arctic_miz_extent_fsdrad.month, wim_arctic_miz_extent_fsdrad/wim_arctic_sie, color=color_fsd, linestyle='--')

axes[0].set_ylabel("Marginal Ice Zone Fraction")

plt.savefig(f"{plotfolder}/MIZ_fraction_all_definitions_ts.png", dpi=600, bbox_inches="tight") 

### Daily outputs

In [ ]:
# Average monthly
df_fraser_all['doy'] = pd.to_datetime(df_fraser_all['date']).dt.dayofyear
df_doy = df_fraser_all.groupby('doy').mean(numeric_only=True)

In [ ]:
# ds_daily = datastore.search(variable=["aice", "wave_sig_ht", "fsdrad"], frequency="1day").to_dask(
#     xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
#         compat="override",
#         data_vars="minimal",
#         coords="minimal",
#     ),
#     xarray_open_kwargs = dict(
#         chunks={"nj": -1, "ni": -1}, # Good for spatial operations, but not temporal
#         decode_timedelta=True
#     )
# )
# ds_grid = datastore.search(variable=["tarea", "HTE"], frequency="fx", realm="seaIce").to_dask().compute()
# ds_grid

# ds_daily = xr.merge([ds_daily, ds_grid])
# ds_daily = ds_daily.isel(time=slice(60,-1))
# ds_daily

In [ ]:
ds_mcw_doy = ds_mcw.groupby('time.dayofyear').mean('time').compute()
ds_mcw_doy

In [ ]:
ds_wim_doy = ds_wim.groupby('time.dayofyear').mean('time').compute()
ds_wim_doy

In [ ]:
hemisphere='south'

mcw_antarctic_miz_width_wave_d = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
mcw_antarctic_miz_width_wave_d_1cm = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', threshold=0.01, hemisphere=hemisphere, freq='dayofyear')
mcw_antarctic_miz_width_wave_d_5cm = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', threshold=0.05, hemisphere=hemisphere, freq='dayofyear')
mcw_antarctic_miz_width_aice_d = calculate_miz_width(ds_mcw_doy, definition='aice', threshold=sic_threshold, hemisphere=hemisphere, freq='dayofyear')
mcw_antarctic_miz_width_fsdrad_d = calculate_miz_width(ds_mcw_doy, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere, freq='dayofyear')

mcw_antarctic_miz_width_wave_d_eff = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')
mcw_antarctic_miz_width_wave_d_eff_1cm = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', threshold=0.01, hemisphere=hemisphere, freq='dayofyear', integration='weighted')
mcw_antarctic_miz_width_wave_d_eff_5cm = calculate_miz_width(ds_mcw_doy, definition='wave_sig_ht', threshold=0.05, hemisphere=hemisphere, freq='dayofyear', integration='weighted')
mcw_antarctic_miz_width_fsdrad_d_eff = calculate_miz_width(ds_mcw_doy, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
wim_antarctic_miz_width_wave_d = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
wim_antarctic_miz_width_wave_d_1cm = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', threshold=0.01, hemisphere=hemisphere, freq='dayofyear')
wim_antarctic_miz_width_wave_d_5cm = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', threshold=0.05, hemisphere=hemisphere, freq='dayofyear')
wim_antarctic_miz_width_aice_d = calculate_miz_width(ds_wim_doy, definition='aice', threshold=sic_threshold, hemisphere=hemisphere, freq='dayofyear')
wim_antarctic_miz_width_fsdrad_d = calculate_miz_width(ds_wim_doy, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere, freq='dayofyear')

wim_antarctic_miz_width_wave_d_eff = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')
wim_antarctic_miz_width_wave_d_eff_1cm = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', threshold=0.01, hemisphere=hemisphere, freq='dayofyear', integration='weighted')
wim_antarctic_miz_width_wave_d_eff_5cm = calculate_miz_width(ds_wim_doy, definition='wave_sig_ht', threshold=0.05, hemisphere=hemisphere, freq='dayofyear', integration='weighted')
wim_antarctic_miz_width_fsdrad_d_eff = calculate_miz_width(ds_wim_doy, definition='fsdrad', threshold=fsd_threshold, hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
# region = 'ANTARCTICA'
fig, axes = plt.subplots(ncols=1, figsize=(7,4), sharex=True, sharey=True)
ax = axes
x_mcw = ds_mcw_doy.dayofyear
x_wim = ds_wim_doy.dayofyear
# ax.plot(x_mcw, mcw_antarctic_miz_width_wave_d_1cm, label='$H_s$ > 1 cm')
# ax.plot(x_mcw, mcw_antarctic_miz_width_wave_d_5cm, label='$H_s$ > 5 cm')
ax.plot(x_mcw, mcw_antarctic_miz_width_wave_d, label='$H_s$ > 30 cm', color=color_swh)
ax.plot(x_wim, wim_antarctic_miz_width_wave_d, color=color_swh, linestyle='--')

ax.plot(x_mcw, mcw_antarctic_miz_width_fsdrad_d, label=f'Floe radii < {fsd_threshold} m', color=color_fsd)
ax.plot(x_wim, wim_antarctic_miz_width_fsdrad_d, color=color_fsd, linestyle='--')

# axes[0].plot(x, antarctic_miz_width_aice_d, label='Ice concentration', color='lightblue')
# axes[0].plot(x, antarctic_miz_width_fsdrad_d, label='Floe size')
ax.plot(range(366), df_doy['mizWidthAlongTrackFromMyEdge'], color='k', label="AltiKa")

ax.legend(frameon=False, ncols=1, fontsize=12)
ax.set_ylim(bottom=0)
ax.set_xlabel("Day of year")
ax.set_ylabel("MIZ width [km]")

plt.savefig(f"{plotfolder}/MIZ_width_comparison_daily.png", dpi=600, bbox_inches="tight") 

In [ ]:
# region = 'ANTARCTICA'
fig, axes = plt.subplots(ncols=1, figsize=(7,4), sharex=True, sharey=True)
ax = axes
# ax.plot(x, antarctic_miz_width_wave_d_eff_1cm, label='$H_s$ > 1 cm')
# ax.plot(x, antarctic_miz_width_wave_d_eff_5cm, label='$H_s$ > 5 cm')
ax.plot(range(366), df_doy['mizWidthAlongTrackFromMyEdge'], color='k', label="AltiKa absolute MIZ width", lw=2)
ax.plot(x_mcw, mcw_antarctic_miz_width_wave_d_eff, label='MCW', color='tab:red', lw=2)
ax.plot(x_wim, wim_antarctic_miz_width_wave_d_eff, label='CICE6-WIM', color='tab:green', linestyle='-')

# ax.plot(x, antarctic_miz_width_fsdrad_d_eff, label=f'Floe radii < {floe_size_threshold} m')
# axes[0].plot(x, antarctic_miz_width_aice_d, label='Ice concentration', color='lightblue')



ax.legend(frameon=False, ncols=1, fontsize=12)
ax.set_ylim(bottom=0)
ax.set_xlabel("Day of year")
ax.set_ylabel("Effective MIZ width (km)")

plt.savefig(f"{plotfolder}/MIZ_width_comparison_daily_effective.png", dpi=600, bbox_inches="tight") 

In [ ]:
mcw_antarctic_miz_width_wave_d_eff = calculate_miz_width(ds_mcw, definition='wave_sig_ht', hemisphere=hemisphere, freq='daily', integration='weighted')
wim_antarctic_miz_width_wave_d_eff = calculate_miz_width(ds_wim, definition='wave_sig_ht', hemisphere=hemisphere, freq='daily', integration='weighted')

In [ ]:
# df_fraser_all['doy'] = pd.to_datetime(df_fraser_all['date']).dt.dayofyear
df_daily = df_fraser_all.groupby('date').mean(numeric_only=True)

In [ ]:
# wim_antarctic_miz_width_wave_d_eff.plot()
mcw_antarctic_miz_width_wave_d_eff.plot()
plt.plot(df_daily['mizWidthAlongTrackFromMyEdge'], color='k', label="AltiKa absolute MIZ width")

In [ ]:
# wim_antarctic_miz_width_wave_d_eff.plot()

plt.plot(df_daily['mizWidthAlongTrackFromMyEdge'], color='k', label="AltiKa absolute MIZ width")
mcw_antarctic_miz_width_wave_d_eff.plot()


In [ ]:
df1.rename("altika").to_frame()

In [ ]:
mcw_antarctic_miz_width_wave_d_eff = calculate_miz_width(ds_mcw, definition='wave_sig_ht', threshold = 0.25, hemisphere=hemisphere, freq='daily', integration='weighted')
# 0.25 = 0.7 = 37.9
# 0.23 = 0.71 = 38.5
# 0.27 = 0.7 = 38.9

In [ ]:
from scipy.stats import pearsonr

# --- Ensure alignment in time ---
df1 = df_daily['mizWidthAlongTrackFromMyEdge']
df2 = mcw_antarctic_miz_width_wave_d_eff.to_series()
df2.index = df2.index.normalize()
# Drop times where either is missing
df = (
    df1.rename("altika")
       .to_frame()
       .join(df2.rename("model"), how="inner")
)

# --- Moving average window (days) ---
window = 30

altika_ma = df["altika"].rolling(window, center=True).mean()
model_ma  = df["model"].rolling(window, center=True).mean()

# --- Pearson R and R^2 ---
r, _ = pearsonr(df["altika"].values, df["model"].values)
r2 = r**2

t = df.index
t_years = t.year + (t.dayofyear - 1) / 365.25

# --- Linear trends ---
# coef_altika = np.polyfit(t_years, df["altika"], 1)
# coef_model  = np.polyfit(t_years, df["model"], 1)
annual = df.resample("YS").mean()

t_years_ann = annual.index.year

coef_altika = np.polyfit(t_years_ann, annual["altika"], 1)
coef_model  = np.polyfit(t_years_ann, annual["model"], 1)

trend_altika = np.polyval(coef_altika, t_years)
trend_model  = np.polyval(coef_model, t_years)

In [ ]:
black = [0.3,0.3,0.3]
orange = np.array([230,159,0])/255
skyblue = np.array([86,180,233])/255
bluishgreen = np.array([0,158,115])/255
yellow = np.array([240,228,66])/255
blue = np.array([0,114,178])/255
vermillion = np.array([213,94,0])/255
reddishpurple = np.array([204,121,167])/255

fig, ax = plt.subplots(figsize=(7, 3))

# Trend lines
# ax.plot(df.index, trend_altika,
#         color=black, ls="--", lw=2,
#         label="")

# ax.plot(df.index, trend_model,
#         color=vermillion, ls="--", lw=2,
#         label="")

# Raw data (transparent)
ax.plot(df.index, df["altika"],
        color=black, alpha=0.3, lw=1,
        label="")

ax.plot(df.index, df["model"],
        color=vermillion, alpha=0.3, lw=1,
        label="")

# Moving averages (bold)
ax.plot(df.index, altika_ma,
        color=black, lw=2.5,
        label=f"AltiKa")

ax.plot(df.index, model_ma,
        color=vermillion, lw=2.5,
        label=f"MOM6–CICE6–WW3")



# Shared x-limits
ax.set_xlim(df.index.min(), df.index.max())

# Labels
ax.set_ylabel("MIZ width (km)")  # adjust if needed
ax.set_xlabel("Time")


from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(df["altika"], df["model"])
)
ax.text(
    0.02, 0.95,
    rf"$R^2 = {r2:.2f}$"
    "\n"
    rf"RMSE = {rmse:.1f} km",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=11
)

ax.legend(frameon=False)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(
    f"{plotfolder}/miz_width_timeseries.png",
    dpi=600,
    bbox_inches="tight",
    transparent=True
)
plt.show()

In [ ]:
coef_altika, coef_model

In [ ]:
plt.plot(range(len(wim_antarctic_miz_width_wave_d_eff.time)), wim_antarctic_miz_width_wave_d_eff)

In [ ]:
plt.plot(range(len(df_daily['mizWidthAlongTrackFromMyEdge'])), df_daily['mizWidthAlongTrackFromMyEdge'], color='k', label="AltiKa absolute MIZ width")

In [ ]:
# plt.figure(figsize=(8, 5))
# plt.scatter(
#     df_all['lonAtMyEdge'],
#     df_all['doy'],
#     c=df_all['mizWidthAlongTrackFromMyEdge'],
#     s=20
# )
# plt.colorbar(label='MIZ Width (km)')
# plt.xlabel('Longitude')
# plt.ylabel('Latitude')
# plt.title('MIZ Width Along Track')
# plt.show()

In [ ]:
# X, Y = np.meshgrid(df_all['lonAtMyEdge'], df_all['doy'])
# Z = df_all['mizWidthAlongTrackFromMyEdge']

# plt.pcolormesh(X, Y, Z, shading='auto')
# plt.colorbar()

In [ ]:
ds_wim.time

In [ ]:
ds_mcw.time

### Hovmoller for one year

In [ ]:
hemisphere = "south"
df_fraser_all.head()

In [ ]:
def fill_nan_with_interpolation(array):
    # Create a meshgrid for the coordinates
    x = np.arange(0, array.shape[1])
    y = np.arange(0, array.shape[0])
    X, Y = np.meshgrid(x, y)
    
    # Extract valid (non-NaN) points and their values
    valid_mask = ~np.isnan(array)
    coords_valid = np.vstack((X[valid_mask], Y[valid_mask])).T
    values_valid = array[valid_mask]

    # Extract coordinates of NaN values to be filled
    coords_to_fill = np.vstack((X[~valid_mask], Y[~valid_mask])).T
    
    # Fill NaN values using griddata interpolation
    array[~valid_mask] = griddata(coords_valid, values_valid, coords_to_fill, method='linear', fill_value=np.nan)
    
    return array

Select one year of data from the Fraser AltiKa dataset

In [ ]:
year = 2020
df_fraser_oneyear = df_fraser_all[df_fraser_all["year"] == year]
df_fraser_oneyear.head()

In [ ]:
df_tmp = df_fraser_oneyear.copy()
df_tmp.set_index('first_meas_time', inplace=True)

hov_data_myedge = np.empty((360, 366),dtype=np.float32)
hov_data_myedge.fill(np. NaN) #Fill the array with Nan values
# hov_data_altikaedge=np.empty((360, 365+366),dtype=np.float32)
# hov_data_altikaedge.fill(np.NaN) #Fill the array with Nan values

# start sprinkling in the data..
for index, row in tqdm(df_tmp.iterrows(), total=len(df_tmp)):
    datetime = index #row['first_meas_time']
    miz_myedge = row['mizWidthAlongTrackFromMyEdge']
    miz_altikaedge = row['mizWidthAlongTrackFromAltikaEdge']
    lon = row['lonAtInnerMIZ']
    doy = pd.to_datetime(datetime).dayofyear

    hov_data_myedge[np.floor(lon).astype(np.int16), doy-1]=miz_myedge
    # hov_data_altikaedge[np.floor(lon).astype(np.int16), doy-1]=miz_altikaedge


In [ ]:
filled_hov_data_myedge = fill_nan_with_interpolation(hov_data_myedge)

In [ ]:
fig = plt.figure(figsize=(10, 13))

lons = np.arange(0, 360)
dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')

# Use gridspec to help size elements of plot; small top plot and big bottom plot
gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[1, 6], hspace=0.03)

# Tick labels
x_tick_labels = [u'0\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}E',
                 u'180\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}W',
                 u'0\N{DEGREE SIGN}E']

# Top plot for geographic reference (makes small map)
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
ax1.set_extent([0, 357.5, -90, -50], ccrs.PlateCarree(central_longitude=180))
ax1.set_yticks([-80, -60])
ax1.set_yticklabels([u'80\N{DEGREE SIGN}S', u'60\N{DEGREE SIGN}S'])
ax1.set_xticks([-180, -90, 0, 90, 180])
ax1.set_xticklabels(x_tick_labels)
ax1.grid(linestyle='dotted', linewidth=2)
COLOR_LAND = (0.7, 0.7, 0.7)
land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                        edgecolor='black', facecolor=COLOR_LAND, linewidth=0.5)
ax1.add_feature(land_50m)

# Add geopolitical boundaries for map reference
ax1.add_feature(cft.COASTLINE.with_scale('50m'))
ax1.add_feature(cft.LAKES.with_scale('50m'), color='black', linewidths=0.5)

##############################################################################
# Bottom plot for Hovmoller diagram
ax2 = fig.add_subplot(gs[1, 0])
ax2.invert_yaxis()  # Reverse the time order to do oldest first
cmap = mpl.cm.viridis
max_pen = 700
norm = mpl.colors.Normalize(vmin=0, vmax=max_pen)

clevs = np.arange(0, max_pen, 50)

cf = ax2.contourf(lons, dates, filled_hov_data_myedge[:,0:365].T, clevs, cmap=cmap, extend='both')

#data_xr = xr.DataArray(wave_penetration_array/1000, dims=["time", "ni"])
#cf = ax2.contourf(lons.roll(ni=-1121), dates, mpcalc.smooth_n_point(
#                  data_xr.roll(ni=-1121), 9, 3), clevs, cmap=cmap, extend='both')
#cs = ax2.contour(lons.roll(ni=-1121), dates, mpcalc.smooth_n_point(
#                 data_xr.roll(ni=-1121), 9, 3), clevs, colors='k', linewidths=0.1)

# Make some ticks and tick labels
ax2.set_xticks([0, 90, 180, 270, 357.5])
ax2.set_xticklabels(x_tick_labels)

ax2.yaxis.set_major_locator(mdates.MonthLocator())
ax2.yaxis.set_major_formatter(mdates.DateFormatter('%b'))

cbar = plt.colorbar(cf, orientation='horizontal', pad=0.04, aspect=50, extendrect=True)
cbar.set_label('MIZ width (km)')

plt.savefig(f"{plotfolder}/Hovmoller_AltiKa_{year}.png", dpi=600, bbox_inches="tight", transparent=True) 

One year of MCW

In [ ]:
def calculate_miz_width_spatial(ds_cice, definition='wave', threshold=0.3, integration='simple', 
                        measure='median', mask=True, hemisphere='south', freq='monthly'):
    mask_2d = _get_mask(ds_cice, definition, hemisphere, threshold)
    
    miz_width_2d = ds_cice['HTE'].where(mask_2d) / 1000 # m to km
    miz_width_1d = _integrate_width(miz_width_2d, ds_cice, integration)
    miz_width_reduced = _reduce_width(miz_width_1d, measure, mask)

    if freq == 'monthly':
        coords = {'month': ds_cice['month']}
    elif freq == 'dayofyear':
        coords = {'dayofyear': ds_cice['dayofyear']}

    return xr.DataArray(
        miz_width_1d,
        dims=miz_width_1d.dims,
        coords=coords,
        name='miz_width',
        attrs={
            'units': 'km',
            'description': f'MIZ width ({definition})'
        }
    )

In [ ]:
year = 2020
ds_mcw_oneyear = ds_mcw.sel(time=ds_mcw.time.dt.year == year)

ds_mcw_oneyear_doy = (
    ds_mcw_oneyear
    .groupby("time.dayofyear")
    .mean("time")
    .compute()
)
ds_mcw_oneyear_doy

In [ ]:
mcw_spatial_miz_width_wave_d = calculate_miz_width_spatial(ds_mcw_oneyear_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
mcw_spatial_miz_width_wave_d_eff = calculate_miz_width_spatial(ds_mcw_oneyear_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
def figure_hovmoller(data, doy):
    fig = plt.figure(figsize=(10, 13))

    lons = ds_mcw_oneyear['TLON'].isel(nj=50)
    dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')[:len(doy)]
    
    # Use gridspec to help size elements of plot; small top plot and big bottom plot
    gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[1, 6], hspace=0.03)
    
    # Tick labels
    x_tick_labels = [u'0\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}E',
                     u'180\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}W',
                     u'0\N{DEGREE SIGN}E']
    
    # Top plot for geographic reference (makes small map)
    ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
    ax1.set_extent([0, 357.5, -90, -50], ccrs.PlateCarree(central_longitude=180))
    ax1.set_yticks([-80, -60])
    ax1.set_yticklabels([u'80\N{DEGREE SIGN}S', u'60\N{DEGREE SIGN}S'])
    ax1.set_xticks([-180, -90, 0, 90, 180])
    ax1.set_xticklabels(x_tick_labels)
    ax1.grid(linestyle='dotted', linewidth=2)
    COLOR_LAND = (0.7, 0.7, 0.7)
    land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                            edgecolor='black', facecolor=COLOR_LAND, linewidth=0.5)
    ax1.add_feature(land_50m)
    
    # Add geopolitical boundaries for map reference
    ax1.add_feature(cft.COASTLINE.with_scale('50m'))
    ax1.add_feature(cft.LAKES.with_scale('50m'), color='black', linewidths=0.5)
    
    ##############################################################################
    # Bottom plot for Hovmoller diagram
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.invert_yaxis()  # Reverse the time order to do oldest first
    cmap = mpl.cm.viridis
    max_pen = 700
    norm = mpl.colors.Normalize(vmin=0, vmax=max_pen)
    
    clevs = np.arange(0, max_pen, 50)
    
    cf = ax2.contourf(lons, dates, data, clevs, cmap=cmap, extend='both')
    
    # Make some ticks and tick labels
    ax2.set_xticks([0, 90, 180, 270, 357.5])
    ax2.set_xticklabels(x_tick_labels)
    
    ax2.yaxis.set_major_locator(mdates.MonthLocator())
    ax2.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
    
    cbar = plt.colorbar(cf, orientation='horizontal', pad=0.04, aspect=50, extendrect=True)
    cbar.set_label('MIZ width (km)')

    axes = [ax1, ax2]

    return fig, axes

In [ ]:
mcw_spatial_miz_width_wave_d

In [ ]:
figure_hovmoller(mcw_spatial_miz_width_wave_d_eff[:365,:], mcw_spatial_miz_width_wave_d_eff.dayofyear)
plt.savefig(f"{plotfolder}/Hovmoller_MCW_{year}.png", dpi=600, bbox_inches="tight", transparent=True) 

One year of CICE6-WIM

In [ ]:
year = 2011
ds_wim_oneyear = ds_wim.sel(time=ds_wim.time.dt.year == year)

ds_wim_oneyear_doy = (
    ds_wim_oneyear
    .groupby("time.dayofyear")
    .mean("time")
    .compute()
)
ds_wim_oneyear_doy

In [ ]:
wim_spatial_miz_width_wave_d = calculate_miz_width_spatial(ds_wim_oneyear_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
wim_spatial_miz_width_wave_d_eff = calculate_miz_width_spatial(ds_wim_oneyear_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
figure_hovmoller(wim_spatial_miz_width_wave_d_eff, wim_spatial_miz_width_wave_d_eff.dayofyear)
plt.savefig(f"{plotfolder}/Hovmoller_WIM_{year}.png", dpi=600, bbox_inches="tight") 

In [ ]:
# ds_wim_oneyear.groupby("time.dayofyear").mean("time").compute()
# ds_wim_oneyear.time.dayofyear

### Hovmoller for a climatology

In [ ]:
df_tmp = df_fraser_all.copy()
df_tmp.set_index('first_meas_time', inplace=True)

In [ ]:
hov_data_myedge = np.empty((360, 366),dtype=np.float32)
hov_data_myedge.fill(np. NaN) #Fill the array with Nan values
# hov_data_altikaedge=np.empty((360, 365+366),dtype=np.float32)
# hov_data_altikaedge.fill(np.NaN) #Fill the array with Nan values

# start sprinkling in the data..
for index, row in df_fraser_all.iterrows():
    datetime = row['first_meas_time']
    miz_myedge = row['mizWidthAlongTrackFromMyEdge']
    miz_altikaedge = row['mizWidthAlongTrackFromAltikaEdge']
    lon = row['lonAtInnerMIZ']
    doy = pd.to_datetime(datetime).dayofyear

    hov_data_myedge[np.floor(lon).astype(np.int16), doy-1]=miz_myedge
    # hov_data_altikaedge[np.floor(lon).astype(np.int16), doy-1]=miz_altikaedge


In [ ]:
# pd.to_datetime(datetime).dayofyear

In [ ]:
# row['first_meas_time']

In [ ]:
filled_hov_data_myedge = fill_nan_with_interpolation(hov_data_myedge)

In [ ]:
# plt.pcolormesh(filled_hov_data_myedge[:,::-1].T, vmin=0, vmax=500)

In [ ]:

fig = plt.figure(figsize=(10, 13))

lons = np.arange(0, 360)
dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')

# Use gridspec to help size elements of plot; small top plot and big bottom plot
gs = gridspec.GridSpec(nrows=2, ncols=1, height_ratios=[1, 6], hspace=0.03)

# Tick labels
x_tick_labels = [u'0\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}E',
                 u'180\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}W',
                 u'0\N{DEGREE SIGN}E']

# Top plot for geographic reference (makes small map)
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
ax1.set_extent([0, 357.5, -90, -50], ccrs.PlateCarree(central_longitude=180))
ax1.set_yticks([-80, -60])
ax1.set_yticklabels([u'80\N{DEGREE SIGN}S', u'60\N{DEGREE SIGN}S'])
ax1.set_xticks([-180, -90, 0, 90, 180])
ax1.set_xticklabels(x_tick_labels)
ax1.grid(linestyle='dotted', linewidth=2)
COLOR_LAND = (0.7, 0.7, 0.7)
land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                        edgecolor='black', facecolor=COLOR_LAND, linewidth=0.5)
ax1.add_feature(land_50m)

# Add geopolitical boundaries for map reference
ax1.add_feature(cft.COASTLINE.with_scale('50m'))
ax1.add_feature(cft.LAKES.with_scale('50m'), color='black', linewidths=0.5)

##############################################################################
# Bottom plot for Hovmoller diagram
ax2 = fig.add_subplot(gs[1, 0])
ax2.invert_yaxis()  # Reverse the time order to do oldest first
cmap = mpl.cm.viridis
max_pen = 700
norm = mpl.colors.Normalize(vmin=0, vmax=max_pen)

clevs = np.arange(0, max_pen, 50)

cf = ax2.contourf(lons, dates, filled_hov_data_myedge[:,0:365].T, clevs, cmap=cmap, extend='both')

#data_xr = xr.DataArray(wave_penetration_array/1000, dims=["time", "ni"])
#cf = ax2.contourf(lons.roll(ni=-1121), dates, mpcalc.smooth_n_point(
#                  data_xr.roll(ni=-1121), 9, 3), clevs, cmap=cmap, extend='both')
#cs = ax2.contour(lons.roll(ni=-1121), dates, mpcalc.smooth_n_point(
#                 data_xr.roll(ni=-1121), 9, 3), clevs, colors='k', linewidths=0.1)

# Make some ticks and tick labels
ax2.set_xticks([0, 90, 180, 270, 357.5])
ax2.set_xticklabels(x_tick_labels)

ax2.yaxis.set_major_locator(mdates.MonthLocator())
ax2.yaxis.set_major_formatter(mdates.DateFormatter('%b'))

cbar = plt.colorbar(cf, orientation='horizontal', pad=0.04, aspect=50, extendrect=True)
cbar.set_label('MIZ width (km)')

plt.savefig(f"{plotfolder}/Hovmoller_AltiKa.png", dpi=600, bbox_inches="tight") 

In [ ]:
time_mean_left.min()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import numpy as np

# --- Figure ---
fig = plt.figure(figsize=(14, 10))  # wider to fit two Hovs

# --- GridSpec: 3 rows, 3 columns ---
gs = gridspec.GridSpec(
    nrows=3, ncols=3,
    height_ratios=[1, 1, 6],
    width_ratios=[6, 6, 1],  # 2 Hovs + right marginal
    hspace=0.05,
    wspace=0.05
)

# --- Map (top, spans left & right Hovs) ---
ax1 = fig.add_subplot(gs[0, 0:2], projection=ccrs.PlateCarree(central_longitude=180))
# ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree(central_longitude=180))
ax1.set_extent([0, 357.5, -90, -50], ccrs.PlateCarree(central_longitude=180))
ax1.set_aspect('auto') 
ax1.set_yticks([-80, -60])
ax1.set_yticklabels([u'80\N{DEGREE SIGN}S', u'60\N{DEGREE SIGN}S'])
ax1.set_xticks([-180, -90, 0, 90, 180])
ax1.set_xticklabels(x_tick_labels)
ax1.grid(linestyle='dotted', linewidth=2)
COLOR_LAND = (0.7, 0.7, 0.7)
land_50m = cft.NaturalEarthFeature('physical', 'land', '50m',
                        edgecolor='black', facecolor=COLOR_LAND, linewidth=0.5)
ax1.add_feature(land_50m)

# Add geopolitical boundaries for map reference
ax1.add_feature(cft.COASTLINE.with_scale('50m'))
ax1.add_feature(cft.LAKES.with_scale('50m'), color='black', linewidths=0.5)


# --- Top panel (temporal mean over longitude, spans both Hovs) ---
ax_top = fig.add_subplot(gs[1, 0:2])

# Hov data: assume shape (time, lon)
data_left = filled_hov_data_myedge[:, :]       # left Hov
data_right = mcw_spatial_miz_width_wave_d_eff[:365]  # right Hov



# Optional: style
ax_top.set_ylabel('Mean MIZ width (km)')
ax_top.grid(alpha=0.3)
ax_top.legend(frameon=False, loc='upper right')

# Compute mean over longitude for each Hov (1D array over lon)
time_mean_left = np.nanmean(data_left, axis=1)   # mean across lon for left Hov
time_mean_right = np.nanmean(data_right, axis=0) # mean across lon for right Hov
lons = np.arange(0, 360)
ax_top.plot(lons[2:-2], time_mean_left[2:-2], color='k', lw=2, label='Left Hov')
ax_top.plot(lons, time_mean_right, color='r', lw=2, label='Right Hov')

# Invert y-axis if you want oldest first
# ax_top.invert_yaxis()  # optional, match Hov orientation
ax_top.set_ylim(0,400)


# --- Left Hovmöller ---
ax2 = fig.add_subplot(gs[2, 0])
lons = np.arange(0, 360)
dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')
cf = ax2.contourf(lons, dates, filled_hov_data_myedge[:,0:365].T, clevs, cmap=cmap, extend='both')

# --- Right Hovmöller (shares y-axis with left) ---
ax3 = fig.add_subplot(gs[2, 1], sharey=ax2)
lons = np.arange(0, 360)
dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')
cf = ax3.contourf(lons, dates, mcw_spatial_miz_width_wave_d_eff[:365], clevs, cmap=cmap, extend='both')

# --- Right marginal (space mean over time) ---
ax_right = fig.add_subplot(gs[2, 2], sharey=ax2)

# Compute mean over longitude for each Hov (1D array over lon)
time_mean_left = np.nanmean(data_left, axis=0)   # mean across lon for left Hov
time_mean_right = np.nanmean(data_right, axis=1) # mean across lon for right Hov
dates = pd.date_range(start="2019-01-01",end="2019-12-31", freq='d')
ax_right.plot(time_mean_left[:365], dates, color='k', lw=2, label='Left Hov')
ax_right.plot(time_mean_right, dates, color='r', lw=2, label='Right Hov')

# --- Grid visualisation / labels (optional) ---
for ax, label in zip(
    [ax1, ax_top, ax2, ax3, ax_right],
    ["MAP", "TOP (lon mean)", "HOV LEFT", "HOV RIGHT", "RIGHT MARGIN"]
):
    ax.set_facecolor("none")
    ax.patch.set_edgecolor("k")
    ax.patch.set_linewidth(1.5)
    ax.text(
        0.5, 0.5, label,
        transform=ax.transAxes,
        ha="center", va="center",
        fontsize=12
    )
    ax.set_xticks([])
    ax.set_yticks([])
ax_top.set_yticks(np.arange(0, 400, 100))
# --- Hovmöller axes adjustments ---
ax2.invert_yaxis()
ax3.invert_yaxis()
ax_right.invert_yaxis()

# Example x-axis ticks for Hovs
x_tick_labels = [u'0\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}E',
                 u'180\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}W',
                 u'0\N{DEGREE SIGN}E']

x_tick_labels2 = ['', u'90\N{DEGREE SIGN}E',
                 u'180\N{DEGREE SIGN}E', u'90\N{DEGREE SIGN}W',
                 u'0\N{DEGREE SIGN}E']


ax2.set_xticks([0, 90, 180, 270, 357.5])
ax2.set_xticklabels(x_tick_labels)
ax3.set_xticks([0, 90, 180, 270, 357.5])
ax3.set_xticklabels(x_tick_labels2)

# Y-axis date formatting for Hovs
dates = np.arange("2019-01-01", "2019-12-31", dtype='datetime64[D]')  # placeholder
ax2.yaxis.set_major_locator(mdates.MonthLocator())
ax2.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax3.yaxis.set_major_locator(mdates.MonthLocator())
ax3.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax3.tick_params(axis='y', labelleft=False)
ax_right.tick_params(axis='y', labelleft=False)
plt.show()

In [ ]:
len(mcw_spatial_miz_width_wave_d_eff.dayofyear[:365])

len(mcw_spatial_miz_width_wave_d_eff)

In [ ]:
mcw_spatial_miz_width_wave_d = calculate_miz_width_spatial(ds_mcw_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
mcw_spatial_miz_width_wave_d_eff = calculate_miz_width_spatial(ds_mcw_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
figure_hovmoller(mcw_spatial_miz_width_wave_d_eff[:365], mcw_spatial_miz_width_wave_d_eff.dayofyear[:365])

plt.savefig(f"{plotfolder}/Hovmoller_MCW_effective.png", dpi=600, bbox_inches="tight", transparent=True)

In [ ]:
figure_hovmoller(mcw_spatial_miz_width_wave_d[:365], mcw_spatial_miz_width_wave_d_eff.dayofyear[:365])
plt.savefig(f"{plotfolder}/Hovmoller_MCW.png", dpi=600, bbox_inches="tight") 

In [ ]:
wim_spatial_miz_width_wave_d_eff.shape

CICE6-WIM Hovmollers

In [ ]:
wim_spatial_miz_width_wave_d = calculate_miz_width_spatial(ds_wim_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear')
wim_spatial_miz_width_wave_d_eff = calculate_miz_width_spatial(ds_wim_doy, definition='wave_sig_ht', hemisphere=hemisphere, freq='dayofyear', integration='weighted')

In [ ]:
figure_hovmoller(wim_spatial_miz_width_wave_d_eff[:365,:], wim_spatial_miz_width_wave_d_eff.dayofyear[:365])
plt.savefig(f"{plotfolder}/Hovmoller_WIM_effective.png", dpi=600, bbox_inches="tight") 

In [ ]:
figure_hovmoller(wim_spatial_miz_width_wave_d[:365,:], wim_spatial_miz_width_wave_d_eff.dayofyear[:365])
plt.savefig(f"{plotfolder}/Hovmoller_WIM.png", dpi=600, bbox_inches="tight") 